In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from plot_utils import COLOR_PALETTE as color_palette, collect, pct_improvement

BACKBONES = {'PatchTST': 'patchtst', 'Moment': 'moment'}
BB_COLOR = {'PatchTST': color_palette[0], 'Moment': color_palette[2]}

In [ ]:
# Summary: % MAE improvement over the univariate backbone, LiMA vs mean pool.
summary = {}
for label, bb in BACKBONES.items():
    means, _, _ = collect([('u', bb), ('lima', f'{bb}_mlpquery_incl'), ('pm', f'{bb}_mlpquery_pm')])
    summary[label] = {'LiMA': pct_improvement(means['u'], means['lima']),
                      'Mean pool': pct_improvement(means['u'], means['pm'])}
    for k, v in summary[label].items():
        print(f'{label:<9} {k:<10} n={len(v):>2}  median={v.median():6.2f}%  improved on {(v > 0).sum()}/{len(v)}')

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fontsize = 24
fig, ax = plt.subplots(figsize=(10, 5))

data, positions, colors, hatches = [], [], [], []
for i, label in enumerate(BACKBONES):
    for j, variant in enumerate(['LiMA', 'Mean pool']):
        data.append(summary[label][variant])
        positions.append(i * 3 + 1 + j * 0.8)
        colors.append(BB_COLOR[label])
        hatches.append('' if variant == 'LiMA' else '//')

bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
for patch, color, hatch in zip(bp['boxes'], colors, hatches):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
    patch.set_hatch(hatch)
for median in bp['medians']:
    median.set_color('black')
    median.set_linewidth(1.5)

ax.set_xticks([])
for pos, sub in zip(positions, ['LiMA', 'Mean pool'] * 2):
    ax.text(pos, -0.03, sub, ha='center', va='top', fontsize=fontsize - 6,
            transform=ax.get_xaxis_transform())
for i, label in enumerate(BACKBONES):
    ax.text(i * 3 + 1.4, -0.14, label, ha='center', va='top', fontsize=fontsize,
            transform=ax.get_xaxis_transform())

ax.axhline(0, color='black', linewidth=1, linestyle='dashed')
plt.ylabel('Percentage Improvement', fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.ylim([-10, 10])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.subplots_adjust(bottom=0.22)
plt.savefig('./summary_boxplot.pdf')

In [ ]:
# Gate ablation: % MAE improvement over the univariate backbone, per gate and
# channel inclusion/exclusion arm.
GATES = ['shared', 'channelwise', 'layerwise', 'layerwise_channelwise', 'mlp', 'mlpquery']
GROUP_LABELS = [r'$\beta$ shared', r'$\beta$ channelwise', r'$\beta$ layerwise',
                r'$\beta$ channelwise' + '\n' + 'layerwise', 'MLP', 'MLP w/ query']

def gate_improvements(bb):
    cols = [('u', bb)] + [(f'{g}_{a}', f'{bb}_{g}_{a}') for g in GATES for a in ['incl', 'excl']]
    means, _, _ = collect(cols)
    return [pct_improvement(means['u'], means[f'{g}_{a}']) for g in GATES for a in ['incl', 'excl']]

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fontsize = 24

for model_name, bb in BACKBONES.items():
    data = gate_improvements(bb)
    fig, ax = plt.subplots(figsize=(20, 10))
    positions = [p for i in range(len(GATES)) for p in (i * 3 + 1, i * 3 + 1.8)]
    bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
    for patch in bp['boxes']:
        patch.set_facecolor(BB_COLOR[model_name])
        patch.set_alpha(0.8)
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(1.5)

    ax.set_xticks([])
    for pos, sub in zip(positions, ['incl.', 'excl.'] * len(GATES)):
        ax.text(pos, -0.03, sub, ha='center', va='top', fontsize=fontsize - 2,
                transform=ax.get_xaxis_transform())
    for i, label in enumerate(GROUP_LABELS):
        ax.text(i * 3 + 1.4, -0.10, label, ha='center', va='top', fontsize=fontsize,
                transform=ax.get_xaxis_transform())

    plt.ylim([-20, 20])
    plt.ylabel('Percentage Improvement', fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.20)
    plt.savefig(f'./{model_name}_infini_ablation_boxplot.pdf')
    plt.show()